In [ ]:
!pip install mediapipe==0.10.21 opencv-python pandas numpy

^C


INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/51.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/51.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/51.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/51.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/51.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/51.0 MB 


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
import mediapipe as mp
import cv2
import os
from google.colab import drive

# --- 1. MOUNT AND SETUP ---
try:
    drive.mount('/content/drive', force_remount=True)
except:
    print("Drive already mounted.")

# Paths
#csv_url = '/content/drive/MyDrive/Cleaned_Csv_Files/cleaned_df_1_to_25.csv'
csv_url = '/content/drive/MyDrive/Cleaned_Csv_Files/created_df.csv'
video_folder = '/content/drive/MyDrive/ASL_Videos_Created'
keypoints_dir = '/content/drive/MyDrive/ASL_Keypoints_Data_10'

os.makedirs(keypoints_dir, exist_ok=True)

# --- 2. DYNAMIC PATH VERIFICATION ---
# This checks if your files have extensions like .mp4, .mkv, etc.
sample_files = os.listdir(video_folder)
extension = ""
if sample_files:
    if "." in sample_files[0]:
        extension = "." + sample_files[0].split(".")[-1]
    print(f"Detected file extension: '{extension}'")
else:
    print("ERROR: Video folder is empty!")

# --- 3. DATA PREPARATION ---
df = pd.read_csv(csv_url, dtype={'gloss': str, 'video_id': str})
target_words = df['gloss'].unique()[:25] # First 25 words
df_filtered = df[df['gloss'].isin(target_words)]
print(f"Processing words: {target_words}")

# --- 4. EXTRACTION LOGIC ---
mp_holistic = mp.solutions.holistic

#3old logic

# def extract_keypoints(results):
    # pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
    # face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(468*3)
    # lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    # rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    # return np.concatenate([pose, face, lh, rh])

##new Logic
# Define essential face indices (Lips and Eyes)
FACE_LANDMARKS_DESIRED = [
    0, 13, 14, 17, 37, 39, 40, 61, 84, 87, 88, 91, 95, 146, 178, 181, 185, 267, 
    269, 270, 291, 308, 314, 317, 318, 321, 324, 375, 402, 405, 409  # Lips
    # You can add Eye indices here if needed
]
def extract_keypoints(results):
    # POSE: Keeping all 33 (132 points)
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
    
    # FACE: Selective extraction (Reduced from 1404 to ~93 points)
    if results.face_landmarks:
        face = np.array([[results.face_landmarks.landmark[i].x, 
                          results.face_landmarks.landmark[i].y, 
                          results.face_landmarks.landmark[i].z] for i in FACE_LANDMARKS_DESIRED]).flatten()
    else:
        face = np.zeros(len(FACE_LANDMARKS_DESIRED) * 3)

    # HANDS: Keeping all (Critical for signs)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    
    return np.concatenate([pose, face, lh, rh])

# --- 5. EXECUTION LOOP ---
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    for index, row in df_filtered.iterrows():
        vid_id, gloss = str(row['video_id']), row['gloss']

        # Folder structure: Keypoints / Word / video_id.npy
        save_path = os.path.join(keypoints_dir, gloss)
        os.makedirs(save_path, exist_ok=True)
        npy_file = os.path.join(save_path, f"{vid_id}.npy")

        if os.path.exists(npy_file):
            continue

        # Construct path using detected extension
        video_path = os.path.join(video_folder, f"{vid_id}{extension}")
        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            print(f"File NOT found: {video_path}")
            continue

        video_sequence = []
        start, end = int(row['frame_start']), int(row['frame_end'])
        if end == -1: end = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        curr = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret or curr > end: break
            if curr >= start:
                image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = holistic.process(image)
                video_sequence.append(extract_keypoints(results))
            curr += 1

        cap.release()
        if video_sequence:
            np.save(npy_file, video_sequence)
            print(f"Successfully saved: {vid_id} ({gloss}) | Frames: {len(video_sequence)}")

print("\n--- Extraction complete for the first 25 words! ---")

/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.7.1, so it will not be used.
  warnings.warn(


Mounted at /content/drive
Detected file extension: '.mp4'
Processing words: ['accident' 'africa' 'all']
Successfully saved: 00618 (accident) | Frames: 27
Successfully saved: 00624 (accident) | Frames: 108
Successfully saved: 00626 (accident) | Frames: 43
Successfully saved: 00629 (accident) | Frames: 148
Successfully saved: 00634 (accident) | Frames: 39
Successfully saved: 00635 (accident) | Frames: 98
Successfully saved: 00639 (accident) | Frames: 104
Successfully saved: 65009 (accident) | Frames: 55
Successfully saved: 01382 (africa) | Frames: 44
Successfully saved: 01383 (africa) | Frames: 145
Successfully saved: 01384 (africa) | Frames: 64
Successfully saved: 01387 (africa) | Frames: 39
Successfully saved: 01388 (africa) | Frames: 122
Successfully saved: 01393 (africa) | Frames: 71
Successfully saved: 01394 (africa) | Frames: 84
Successfully saved: 01395 (africa) | Frames: 82
Successfully saved: 01398 (africa) | Frames: 77
Successfully saved: 65029 (africa) | Frames: 65
Successfull

In [ ]:
# import os
# import numpy as np

# # Set your desired sequence length (e.g., 30 frames is standard for ASL)
# SEQUENCE_LENGTH = 25

# def standardize_sequences(keypoints_dir):
#     for word in os.listdir(keypoints_dir):
#         word_path = os.path.join(keypoints_dir, word)
#         if not os.path.isdir(word_path): continue

#         for npy_file in os.listdir(word_path):
#             file_path = os.path.join(word_path, npy_file)
#             window = np.load(file_path)

#             # Scenario A: Video is too long -> Truncate
#             if len(window) > SEQUENCE_LENGTH:
#                 standardized = window[:SEQUENCE_LENGTH]

#             # Scenario B: Video is too short -> Pad with zeros
#             else:
#                 padding = np.zeros((SEQUENCE_LENGTH - len(window), 1662))
#                 standardized = np.concatenate([window, padding], axis=0)

#             # Overwrite the file with the new standardized version
#             np.save(file_path, standardized)
#             print(f"Standardized {npy_file} to {standardized.shape}")

# # Run the standardization
# standardize_sequences(keypoints_dir)

In [ ]:
#changed standarization logic to linear resampling for better performance and accuracy
import os
import numpy as np

SEQUENCE_LENGTH =25

def standardize_sequences(keypoints_dir):
    for word in os.listdir(keypoints_dir):
        word_path = os.path.join(keypoints_dir, word)
        if not os.path.isdir(word_path): continue

        for npy_file in os.listdir(word_path):
            if not npy_file.endswith('.npy'): continue

            file_path = os.path.join(word_path, npy_file)
            window = np.load(file_path)

            # Get the current number of frames
            current_length = len(window)

            if current_length == SEQUENCE_LENGTH:
                standardized = window
            else:
                # LINEAR RESAMPLING LOGIC:
                # We create 25 evenly spaced indices between 0 and the last frame
                indices = np.linspace(0, current_length - 1, SEQUENCE_LENGTH).astype(int)
                standardized = window[indices]

            # Overwrite with the perfectly scaled 25-frame version
            np.save(file_path, standardized)
            print(f"Resampled {word}/{npy_file}: {current_length} -> {standardized.shape}")

# Run the standardization
standardize_sequences(keypoints_dir)